In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

由于卷积块前后需要相加，因此输入与输出的卷积块的通道数，高宽需要一致。如果不一致就通过1*1的卷积块进行调整

这里是为了适配ResNet18，所以每个Resdual残差块是两层

In [5]:
class Residual(nn.Module):
    def __init__(self, input_channels,num_channels,use_1x1conv = False,strides = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels,num_channels,kernel_size=3,padding=1,stride=strides)
        self.conv2 = nn.Conv2d(num_channels,num_channels,kernel_size=3,padding=1)

        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels,num_channels,kernel_size=1,stride= strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))  #第一层是卷积，归一化，激活
        Y = self.bn2(self.conv2(Y))  #第二层先卷积，归一化，然后与残差相加后再激活

        #如果通道数不一致那么使用1*1卷积之后再相加
        if self.conv3:
            X = self.conv3(X)
        #残差连接
        Y +=X
        return F.relu(Y)

In [6]:
blk = Residual(3,3)  #告诉网络输入3通道的数据，输出也是三通道（RGB）
X = torch.rand(4,3,6,6) #Batch,Channel,Height,Width
Y = blk(X)
Y.shape

torch.Size([4, 3, 6, 6])

In [7]:
blk = Residual(3,6,use_1x1conv=True,strides=2)
blk(X).shape

torch.Size([4, 6, 3, 3])

1. 需要掌握给定输入大小，计算出输出大小的能力，比如现在就是stride为2的时候那么宽高就要减半，如果是1就不变，然后根据Residual(3,6,use_1x1conv=True,strides=2)得出输出是6通道，输入是3通道
2. 手绘出ResNet的结构图

In [3]:
b1 = nn.Sequential(nn.Conv2d(1,64,kernel_size=7,stride=2,padding=3), #快速把通道数从1变成64，使用7x7大卷积核
                   nn.BatchNorm2d(64),nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3,stride=2,padding=1)) #再次把尺寸减半

块也就是 Residual 类的实例。num_residuals 决定了一个阶段里有多少个块。

In [ ]:
def resnet_block(input_channels,num_channels,num_residuals,first_block = False):
    blk = []
    for i in range(num_residuals): #每个阶段有多少块
        if i == 0 and not first_block:
            blk.append(Residual(input_channels,num_channels,use_1x1conv=True,strides=2)) #第一块的第一层不需要再缩小，其余块的第一层缩小然后进行提取
        else:
            blk.append(Residual(num_channels,num_channels))
    return blk

In [9]:
b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True)) #声明出来这个是第一个块
b3 = nn.Sequential(*resnet_block(64, 128, 2))
b4 = nn.Sequential(*resnet_block(128, 256, 2))
b5 = nn.Sequential(*resnet_block(256, 512, 2))

最后假如全局平均汇聚层，以及全连接输出

In [10]:
net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool2d((1,1)),
                    nn.Flatten(), nn.Linear(512, 10))

In [11]:
X = torch.rand(size=(1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 128, 28, 28])
Sequential output shape:	 torch.Size([1, 256, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 512, 1, 1])
Flatten output shape:	 torch.Size([1, 512])
Linear output shape:	 torch.Size([1, 10])
